# Notebook 06: vLLM serving sweep with tensor parallelism across both GPUs

**Scenario:** vLLM splits every layer across the two GPUs (`tensor_parallel_size=2`), so both GPUs work on every token at once. This is real multi-GPU serving and gives the throughput numbers for H3.

**What it measures**

1. Throughput vs. batch size for **stock DiffusionGemma** and **Gemma 4 AR** (bf16, same GPUs, same prompts): the crossover curve.
2. Optionally, a **PyTorch profiler trace** of DiffusionGemma inside vLLM, to see how much of each step is sampling.

**Before you start**
- Use a **separate Python environment** for vLLM; it pins its own torch and transformers versions and can break the other notebooks' environment.
- Check the GPU link: tensor parallelism exchanges data twice per layer, so NVLink vs. PCIe changes the numbers. Report which one you have.
- Each model runs in its **own subprocess**, so GPU memory is fully released between runs.

## 1. Hardware and install

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!nvidia-smi topo -m

In [ ]:
# Run in a fresh virtual environment, e.g.:
#   python -m venv ~/venv-vllm && source ~/venv-vllm/bin/activate && pip install -U vllm pandas matplotlib jupyter
import importlib.util
print('vllm installed:', importlib.util.find_spec('vllm') is not None)

## 2. The sweep script

In [ ]:
%%writefile vllm_sweep.py
"""Throughput sweep for one model in vLLM. Usage:
python vllm_sweep.py --model ID --tp 2 --out results.json [--profile DIR]"""
import argparse, json, os, time

p = argparse.ArgumentParser()
p.add_argument('--model', required=True)
p.add_argument('--tp', type=int, default=2)
p.add_argument('--dtype', default='bfloat16')
p.add_argument('--batches', default='1,2,4,8,16,32,64')
p.add_argument('--max-tokens', type=int, default=512)
p.add_argument('--max-model-len', type=int, default=4096)
p.add_argument('--ignore-eos', action='store_true', help='force fixed-length outputs (check it is supported for the model)')
p.add_argument('--profile', default=None, help='directory for a torch profiler trace of one batch-8 run')
p.add_argument('--out', required=True)
a = p.parse_args()
if a.profile:
    os.environ['VLLM_TORCH_PROFILER_DIR'] = a.profile

from vllm import LLM, SamplingParams

llm = LLM(model=a.model, tensor_parallel_size=a.tp, dtype=a.dtype,
          max_model_len=a.max_model_len, gpu_memory_utilization=0.90)
sp = SamplingParams(max_tokens=a.max_tokens, temperature=0.0, ignore_eos=a.ignore_eos)
base_prompts = [
    'Explain how a rainbow forms, in detail.',
    'Write a Python function that merges two sorted lists, with comments.',
    'Describe the water cycle for a high-school student.',
    'Summarize the causes of the French Revolution.',
]
llm.generate(base_prompts[:2], sp)                     # warm-up

results = []
for B in [int(x) for x in a.batches.split(',')]:
    prompts = [base_prompts[i % len(base_prompts)] for i in range(B)]
    t0 = time.time()
    outs = llm.generate(prompts, sp)
    dt = time.time() - t0
    n_tok = sum(len(o.outputs[0].token_ids) for o in outs)
    results.append(dict(model=a.model, batch=B, seconds=dt, total_tokens=n_tok,
                        total_tok_per_s=n_tok / dt, per_request_tok_per_s=n_tok / dt / B,
                        mean_output_tokens=n_tok / B))
    print(results[-1], flush=True)

if a.profile:
    prompts = [base_prompts[i % len(base_prompts)] for i in range(8)]
    llm.start_profile(); llm.generate(prompts, sp); llm.stop_profile()
    print('profiler trace written to', a.profile)

json.dump(results, open(a.out, 'w'), indent=1)

## 3. Run both models

Model ids: the DiffusionGemma id is the official one. **Confirm the Gemma 4 AR id on Hugging Face** (the same-size instruct model, 26B A4B) before running. Each run loads the model fresh, so expect several minutes of loading per model.

In [ ]:
DIFF_ID = 'google/diffusiongemma-26B-A4B-it'
AR_ID   = 'google/gemma-4-26b-a4b-it'      # CONFIRM on Hugging Face
OUT = './runs/nb06'
import os; os.makedirs(OUT, exist_ok=True)

In [ ]:
!python vllm_sweep.py --model {DIFF_ID} --tp 2 --out {OUT}/diffusion.json

In [ ]:
!python vllm_sweep.py --model {AR_ID} --tp 2 --out {OUT}/ar.json

## 4. Crossover plot

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
df = pd.concat([pd.DataFrame(json.load(open(f'{OUT}/diffusion.json'))).assign(kind='DiffusionGemma'),
                pd.DataFrame(json.load(open(f'{OUT}/ar.json'))).assign(kind='Gemma 4 AR')])
display(df[['kind', 'batch', 'total_tok_per_s', 'per_request_tok_per_s', 'mean_output_tokens']].round(1))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for kind, g in df.groupby('kind'):
    ax[0].plot(g.batch, g.total_tok_per_s, marker='o', label=kind)
    ax[1].plot(g.batch, g.per_request_tok_per_s, marker='o', label=kind)
ax[0].set(xscale='log', xlabel='batch size', ylabel='total tokens/s', title='Total throughput')
ax[1].set(xscale='log', xlabel='batch size', ylabel='tokens/s per request', title='Per-request speed')
for x in ax: x.grid(alpha=0.3); x.legend()
plt.tight_layout(); plt.savefig(f'{OUT}/crossover.png', dpi=150); plt.show()

**Reading it:** the batch size where Gemma 4 AR's total throughput overtakes DiffusionGemma's is the crossover. The paper reports about 32 on one H100 in FP8; yours will differ (two A100s, bf16, tensor parallel), so report your own and compare the trend, not the exact number. Also check `mean_output_tokens`: if the two models write very different lengths, compare tokens/s rather than seconds.

## 5. Optional: profile DiffusionGemma inside vLLM

Records a PyTorch profiler trace of one batch-8 generation. Open the trace in TensorBoard or `chrome://tracing` / Perfetto and add up time in sampling-related kernels (softmax, top-k, multinomial, the probabilities x embeddings matmul) versus the rest of each step. That share is the most direct motivation number for the paper.

In [ ]:
!python vllm_sweep.py --model {DIFF_ID} --tp 2 --batches 1 --profile {OUT}/trace --out {OUT}/diffusion_profiled.json
!ls -la {OUT}/trace